# Multi-image FLAME head fit with Pixel3DMM

Personal non-commercial research workflow, owner decision 2026-10-02.
Read [Pixel3DMM](https://github.com/SimonGiebenhain/pixel3dmm),
[MICA](https://github.com/Zielon/MICA) and your [FLAME](https://flame.is.tue.mpg.de/)
licence before using their assets. Pixel3DMM is CC BY-NC 4.0; MICA,
insightface models and NVIDIA's renderer have separate restricted terms.
No models or upstream source are embedded here or redistributed by this notebook.

Register at FLAME, download **FLAME2020.zip** yourself and put it in your
Google Drive. Choose **Runtime > Change runtime type > GPU** (T4 works; L4/A100 are faster).
Edit the parameter cell, then **Runtime > Run all**.
A separate Python 3.9 / CUDA 11.8 conda environment avoids restarting Colab.
Installation/native compilation and multi-GB weight downloads can take tens of minutes.
**Colab execution has not been tested.**

Drive is mounted with read-only **usage**, not a read-only permission grant:
Colab's Drive mount can write. This notebook only reads the selected FLAME
archive(s), copies them into the VM and immediately unmounts Drive.
Never place/upload your photos in Drive for this workflow. When prompted,
select de-glassed `front.png`, `left.png`, `right.png`, `back.png` together
(.jpg/.jpeg also accepted). Left/right mean the subject's left/right profile.
Back is optional and omitted from fitting if no face is detected.
Front and both profiles must preprocess successfully.

Photos run on Google's VM only, through user-operated uploads. No remote
inference service or logging service receives them. No previews are displayed.
Neutral head OBJ/PLY, JSON/NPZ parameters, cameras, fitted-view meshes and PNG
overlays download as **head_fit.zip**. Keep it locally in
**user-data/twin/head/flame/** (gitignored); never ship it with the web app.
This is a head intermediate, not the rigged `twin.glb` bundle.

Installation, upload, fitting and download share a `try/finally` cleanup.
The final privacy cell also checks that no session directory remains.
Save the browser download, then **Runtime > Disconnect and delete runtime**.
VM deletion is not secure erasure or a promise about Google's retention.
Clear notebook outputs before sharing. On hard kernel/VM termination,
cleanup cannot run; delete the runtime yourself.


In [ ]:
# Parameters: edit before Runtime > Run all.
FLAME_ZIP_DRIVE_PATH = "/content/drive/MyDrive/FLAME/FLAME2020.zip"
FLAME_VERSION = "2020"  # "2020" (default) or "2023" (no-jaw variant)
# For 2023, point FLAME_ZIP_DRIVE_PATH to FLAME2023.zip AND provide 2020:
# preprocessing/MICA and tracking landmark/mask assets still require 2020.
FLAME2020_ZIP_DRIVE_PATH = "/content/drive/MyDrive/FLAME/FLAME2020.zip"
ITERS = 1500
GLOBAL_ITERS = 1500


## GPU check

Fails before Drive access or image uploads unless a T4/L4/A100/H100 GPU is attached.
These GPU choices are notebook policy; no VRAM/performance guarantee is made.
FLAME 2023 fitting uses `use_flame2023=True` and `ignore_mica=True`.
The MICA stage still runs with its own FLAME 2020 model.


In [ ]:
import subprocess

if FLAME_VERSION not in {"2020", "2023"}:
    raise ValueError("FLAME_VERSION must be 2020 or 2023")
if any(not isinstance(value, int) or value < 1 for value in (ITERS, GLOBAL_ITERS)):
    raise ValueError("Iteration counts must be positive integers")
try:
    gpu_summary = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True
    ).splitlines()[0]
except (FileNotFoundError, subprocess.CalledProcessError, IndexError) as error:
    raise RuntimeError("Select an L4 or A100 Colab GPU runtime") from error
gpu_name, gpu_memory = gpu_summary.rsplit(",", 1)
print(f"GPU: {gpu_name.strip()}, {int(gpu_memory)} MiB")
# T4 (16 GB, sm_75) is the only GPU on Google AI Pro; L4/A100/H100 need AI Ultra.
GPU_ARCHES = {"T4": "7.5", "L4": "8.9", "A100": "8.0", "H100": "9.0"}
GPU_ARCH = next((arch for name, arch in GPU_ARCHES.items() if name in gpu_name), None)
if GPU_ARCH is None or int(gpu_memory) < 15000:
    raise RuntimeError("A T4, L4, A100 or H100 GPU runtime is required")
if GPU_ARCH == "7.5":
    print("T4 detected: fitting is slower; if it runs out of memory, lower ITERS/GLOBAL_ITERS.")


In [ ]:
EMBEDDED_FILES = {'io_utils.py': '"""Original archive/privacy helpers; no third-party model code or data embedded."""\n\nfrom pathlib import Path\nimport gc\nimport shutil\nimport stat\nimport zipfile\n\n\ndef safe_extract(archive: Path, destination: Path) -> None:\n    """Reject traversal and symlinks before extracting a user-supplied model zip."""\n    destination = destination.resolve()\n    with zipfile.ZipFile(archive) as bundle:\n        for item in bundle.infolist():\n            target = (destination / item.filename).resolve()\n            if not target.is_relative_to(destination) or "\\\\" in item.filename:\n                raise ValueError("Unsafe zip member path")\n            if stat.S_ISLNK(item.external_attr >> 16):\n                raise ValueError("Zip symlinks are not allowed")\n        bundle.extractall(destination)\n\n\ndef stage_flame(archive: Path, assets: Path, version: str) -> None:\n    if version not in {"2020", "2023"}:\n        raise ValueError("FLAME_VERSION must be 2020 or 2023")\n    unpacked = assets / ("unpacked-" + version)\n    safe_extract(archive, unpacked)\n    filename = "generic_model.pkl" if version == "2020" else "flame2023_no_jaw.pkl"\n    candidates = list(unpacked.rglob(filename))\n    if len(candidates) != 1:\n        raise ValueError(f"FLAME {version} zip must contain exactly one {filename}")\n    target = assets / ("FLAME" + version)\n    target.mkdir(parents=True, exist_ok=True)\n    # Copy the model\'s sibling files, preserving upstream-supplied landmark assets.\n    shutil.copytree(candidates[0].parent, target, dirs_exist_ok=True)\n    shutil.rmtree(unpacked)\n    if version == "2020":\n        required = ["generic_model.pkl", "landmark_embedding.npy", "FLAME_masks/FLAME_masks.pkl"]\n        missing = [name for name in required if not (target / name).is_file()]\n        if missing:\n            raise ValueError("FLAME 2020 model/landmark/mask assets missing: " + ", ".join(missing))\n\n\ndef upload_views(names) -> dict:\n    result = {}\n    for name in names:\n        path = Path(name)\n        view = path.stem.lower()\n        if view not in {"front", "left", "right", "back"} or path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:\n            raise ValueError("Use front/left/right/back with .png, .jpg or .jpeg extensions")\n        if view in result:\n            raise ValueError("Only one image per view is allowed")\n        result[view] = path.name\n    if not {"front", "left", "right"}.issubset(result):\n        raise ValueError("Front, left profile and right profile are required; back is optional")\n    return result\n\n\ndef cleanup_session(path: Path, parent: Path = Path("/content")) -> None:\n    resolved = path.resolve()\n    if resolved.parent != parent.resolve() or not resolved.name.startswith("dt-pixel3dmm-session-"):\n        raise ValueError("Refusing cleanup outside the private Pixel3DMM session")\n    if resolved.exists():\n        shutil.rmtree(resolved)\n    gc.collect()\n\n\ndef download_and_wait(path: Path) -> None:\n    """Same completion convention as the TRELLIS notebook: await browser memory."""\n    import IPython.display\n    from google.colab import files, output\n    from unittest.mock import patch\n\n    original = IPython.display.display\n    instrumented = False\n\n    def display_with_completion(*objects, **kwargs):\n        nonlocal instrumented\n        changed = []\n        for item in objects:\n            if isinstance(item, IPython.display.Javascript) and item.data.strip().startswith("download("):\n                item = IPython.display.Javascript("window.dtHeadDownload = " + item.data)\n                instrumented = True\n            changed.append(item)\n        return original(*changed, **kwargs)\n\n    with patch.object(IPython.display, "display", display_with_completion):\n        files.download(str(path))\n    if not instrumented:\n        raise RuntimeError("Colab download API changed; completion wrapper needs updating")\n    output.eval_js("window.dtHeadDownload.then(() => true)", timeout_sec=600)\n', 'camera.py': '"""Numeric camera export for Pixel3DMM\'s OpenGL pinhole model (synthetic-testable)."""\n\nimport numpy as np\n\n\ndef camera_matrices(checkpoint: dict) -> tuple:\n    size = np.asarray(checkpoint["img_size"], dtype=int).reshape(2)\n    if size[0] != size[1]:\n        raise ValueError("This adapter requires square cropped images")\n    side = int(size[0])\n    camera = checkpoint["camera"]\n    focal = float(np.asarray(camera["fl"]).reshape(-1)[0]) * side\n    pp = np.asarray(camera["pp"]).reshape(-1, 2)[0]\n    center = (1 + pp) * (side / 2 + 0.5)\n    intrinsics = np.array([[focal, 0, center[0]], [0, focal, center[1]], [0, 0, 1]])\n    base = np.eye(4)\n    base[:3, :3] = np.asarray(camera["R_base_0"]).reshape(-1, 3, 3)[0]\n    base[:3, 3] = np.asarray(camera["t_base_0"]).reshape(-1, 3)[0]\n    head = np.eye(4)\n    head[:3, :3] = np.asarray(checkpoint["flame"]["R_rotation_matrix"]).reshape(-1, 3, 3)[0]\n    head[:3, 3] = np.asarray(checkpoint["flame"]["t"]).reshape(-1, 3)[0]\n    extrinsics = base @ head\n    if not np.isfinite(intrinsics).all() or not np.isfinite(extrinsics).all() or focal <= 0:\n        raise ValueError("Invalid fitted camera")\n    return intrinsics, extrinsics, side\n\n\ndef projection_matrix(intrinsics: np.ndarray, side: int) -> np.ndarray:\n    """Near/far = 0.1/5 m, matching upstream tracking\'s rasterizer.\n\n    Pixel-array row 0 corresponds to NDC y=-1: negative fy is deliberate.\n    This matrix uses pixel edge coordinates; pixel i\'s center is i+0.5.\n    """\n    near, far = 0.1, 5.0\n    result = np.zeros((4, 4), dtype=np.float32)\n    result[0, 0] = 2 * intrinsics[0, 0] / side\n    result[1, 1] = -2 * intrinsics[1, 1] / side\n    result[0, 2] = 1 - 2 * intrinsics[0, 2] / side\n    result[1, 2] = 1 - 2 * intrinsics[1, 2] / side\n    result[2, 2] = -(far + near) / (far - near)\n    result[2, 3] = -2 * far * near / (far - near)\n    result[3, 2] = -1\n    return result\n', 'setup_runtime.py': '"""Colab-only installation adapter. Downloads licensed sources/assets into the VM.\n\nNo upstream code, model, credentials or personal data belongs in this repository.\nSources checked 2026-10-02; full CUDA/Colab execution remains untested.\n"""\n\nfrom pathlib import Path\nimport os\nimport shutil\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport urllib.request\n\nfrom io_utils import safe_extract, stage_flame\n\nPIXEL_REVISION = "fcd1fa973c7715b02a8948dfc679dff53cf85924"\nMICA_REVISION = "af22e7a5810d474bc28a1433db533723d6bd2b07"\nFACER_REVISION = "ddd35c76ff840174b8a5403ad1c1255e37b8782b"\nPIPNET_REVISION = "b9eab58816437403a34aa5bc3adeafe5081fd36b"\nPYTORCH3D_REVISION = "75ebeeaea0908c5527e7b1e305fbc7681382db47"  # stable\nNVDIFFRAST_REVISION = "729261dc64c4241ea36efda84fbf532cc8b425b8"  # v0.3.3\n\n\ndef run(command, root: Path, cwd=None, env=None) -> None:\n    """Keep upstream numeric/debug output in private VM logs, never notebook outputs."""\n    log = root / "runtime.log"\n    with log.open("a", encoding="utf-8") as stream:\n        # Keep the worker and nested stages in one group so interruption kills\n        # subprocesses before the notebook deletes their private directories.\n        effective_env = os.environ if env is None else env\n        new_group = os.name == "posix" and effective_env.get("DT_WORKER_ACTIVE") != "1"\n        process = subprocess.Popen([str(arg) for arg in command], cwd=cwd, env=env,\n                                   stdout=stream, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,\n                                   start_new_session=new_group)\n        try:\n            status = process.wait()\n            if status:\n                raise subprocess.CalledProcessError(status, [str(arg) for arg in command])\n        except BaseException:\n            try:\n                if new_group:\n                    os.killpg(process.pid, signal.SIGKILL)\n                elif process.poll() is None:\n                    process.kill()\n            except ProcessLookupError:\n                pass\n            process.wait()\n            raise\n\n\ndef replace_exact(path: Path, old: str, new: str) -> None:\n    source = path.read_text(encoding="utf-8")\n    if source.count(old) != 1:\n        raise RuntimeError(f"Upstream adapter mismatch in {path.name}; refusing silent fallback")\n    path.write_text(source.replace(old, new), encoding="utf-8")\n\n\ndef mica_assets(root: Path) -> None:\n    """Called instead of upstream\'s credential-based MICA/FLAME install script."""\n    os.environ["DT_WORKER_ACTIVE"] = "1"\n    source = root / "pixel3dmm"\n    mica = source / "src/pixel3dmm/preprocessing/MICA"\n    models = root / "insightface/models"\n    destinations = [\n        ("1bYsI_spptzyuFmfLYqYkcJA6GZWZViNt", mica / "data/pretrained/mica.tar"),\n        ("16PWKI_RjjbE4_kqpElG-YFqe8FpXjads", models / "antelopev2.zip"),\n        ("1navJMy0DTr1_DHjLWu1i48owCPvXWfYc", models / "buffalo_l.zip"),\n    ]\n    for identifier, path in destinations:\n        path.parent.mkdir(parents=True, exist_ok=True)\n        run([sys.executable, "-m", "gdown", identifier, "-O", path], root)\n        if path.suffix == ".zip":\n            target = path.with_suffix("")\n            safe_extract(path, target)\n            # Accommodate either flat or named-folder archives expected by insightface.\n            nested = target / target.name\n            if nested.is_dir():\n                for child in nested.iterdir():\n                    shutil.move(str(child), target / child.name)\n                nested.rmdir()\n    detector = mica / "utils/landmark_detector.py"\n    replace_exact(detector, "FaceAnalysis(name=\'antelopev2\', providers=[\'CUDAExecutionProvider\'])",\n                  "FaceAnalysis(name=\'antelopev2\', root=os.environ[\'DT_INSIGHTFACE_ROOT\'], providers=[\'CPUExecutionProvider\'])")\n    detector.write_text("import os\\n" + detector.read_text(encoding="utf-8"), encoding="utf-8")\n\n\ndef install(root: Path, architecture: str) -> tuple:\n    """Python 3.9 + CUDA 11.8 isolated from the Colab kernel; no restart needed."""\n    print("Installing isolated Python 3.9 / CUDA 11.8; native builds can take tens of minutes.")\n    archive = root / "micromamba.tar.bz2"\n    urllib.request.urlretrieve("https://micro.mamba.pm/api/micromamba/linux-64/2.0.5", archive)\n    with tarfile.open(archive) as bundle:\n        member = bundle.getmember("bin/micromamba")\n        if not member.isfile():\n            raise RuntimeError("Unexpected micromamba package")\n        with bundle.extractfile(member) as binary, (root / "micromamba").open("wb") as output:\n            shutil.copyfileobj(binary, output)\n    mamba = root / "micromamba"\n    mamba.chmod(0o755)\n    prefix = root / "env"\n    cuda_packages = ["cuda-nvcc", "cuda-cccl", "cuda-cudart", "cuda-cudart-dev",\n                     "libcusparse", "libcusparse-dev", "libcublas", "libcublas-dev",\n                     "libcurand", "libcurand-dev", "libcusolver", "libcusolver-dev"]\n    run([mamba, "create", "-y", "--no-rc", "-r", root / "mamba", "-p", prefix,\n         "-c", "conda-forge", "-c", "nvidia/label/cuda-11.8.0",\n         "python=3.9", "pip", "gcc_linux-64=11", "gxx_linux-64=11",\n         *["nvidia/label/cuda-11.8.0::" + package for package in cuda_packages]], root)\n    python = prefix / "bin/python"\n    env = os.environ.copy()\n    env.update({\n        "PATH": str(prefix / "bin") + os.pathsep + env["PATH"],\n        "CONDA_PREFIX": str(prefix), "CUDA_HOME": str(prefix),\n        "LD_LIBRARY_PATH": str(prefix / "lib") + os.pathsep + env.get("LD_LIBRARY_PATH", ""),\n        "CC": str(prefix / "bin/x86_64-conda-linux-gnu-gcc"),\n        "CXX": str(prefix / "bin/x86_64-conda-linux-gnu-g++"),\n        "CUDAHOSTCXX": str(prefix / "bin/x86_64-conda-linux-gnu-g++"),\n        "TORCH_CUDA_ARCH_LIST": architecture, "MAX_JOBS": "2",\n        "TORCH_HOME": str(root / "cache/torch"), "HF_HOME": str(root / "cache/hf"),\n        "XDG_CACHE_HOME": str(root / "cache"), "MPLCONFIGDIR": str(root / "cache/matplotlib"),\n        "TORCH_EXTENSIONS_DIR": str(root / "cache/extensions"),\n        "PIP_CACHE_DIR": str(root / "cache/pip"), "DT_INSIGHTFACE_ROOT": str(root / "insightface"),\n        "WANDB_MODE": "disabled", "HF_HUB_DISABLE_TELEMETRY": "1", "DO_NOT_TRACK": "1",\n        # Only trusted, pinned upstream weights are deserialized in this VM.\n        "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",\n        "MPLBACKEND": "Agg", "DT_PYTHON": str(python), "DT_SESSION": str(root),\n    })\n\n    def pip(*args):\n        run([python, "-m", "pip", "install", *args], root, env=env)\n\n    run(["apt-get", "update", "-qq"], root)\n    run(["apt-get", "install", "-y", "git", "build-essential", "ffmpeg", "libgl1",\n         "libegl1-mesa-dev", "libgles2-mesa-dev"], root)\n    pip("pip==25.1.1", "setuptools==68.2.2", "wheel", "numpy==1.23.5", "scipy==1.11.4", "Cython<3")\n    pip("torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",\n        "--index-url", "https://download.pytorch.org/whl/cu118")\n    source = root / "pixel3dmm"\n    run(["git", "clone", "https://github.com/SimonGiebenhain/pixel3dmm.git", source], root, env=env)\n    run(["git", "-C", source, "checkout", "--detach", PIXEL_REVISION], root, env=env)\n    constraints = root / "constraints.txt"\n    constraints.write_text(\n        "numpy==1.23.5\\nscipy==1.11.4\\nscikit-image==0.22.0\\nopencv-python==4.10.0.84\\n"\n        "matplotlib==3.7.5\\npytorch-lightning==2.5.2\\ntorchmetrics==1.7.2\\n"\n        "librosa==0.10.2.post1\\nnumba==0.60.0\\ntimm==1.0.15\\n"\n        "torch==2.7.1\\ntorchvision==0.22.1\\n", encoding="utf-8")\n    env["PIP_CONSTRAINT"] = str(constraints)\n    requirements = root / "requirements.txt"\n    requirements.write_text((source / "requirements.txt").read_text().replace("numpy==1.23", "numpy==1.23.5"))\n    pip("chumpy==0.70", "--no-build-isolation")\n    pip("-r", str(requirements), "gdown==5.2.0", "trimesh>=4.6,<5", "onnxruntime==1.19.2", "ninja", "iopath")\n    for repo, revision in [("facebookresearch/pytorch3d", PYTORCH3D_REVISION), ("NVlabs/nvdiffrast", NVDIFFRAST_REVISION)]:\n        pip("git+https://github.com/" + repo + ".git@" + revision, "--no-build-isolation")\n    pip("-e", str(source))\n\n    # Execute upstream\'s installer after narrowly adapting its VM copy. Do not\n    # execute the credential prompts or its FLAME downloads at any point.\n    script = source / "install_preprocessing_pipeline.sh"\n    for old, new in [\n        ("git clone git@github.com:FacePerceiver/facer.git", "git clone https://github.com/FacePerceiver/facer.git\\ngit -C facer checkout --detach " + FACER_REVISION),\n        ("git clone git@github.com:Zielon/MICA.git", "git clone https://github.com/Zielon/MICA.git\\ngit -C MICA checkout --detach " + MICA_REVISION),\n        ("git clone https://github.com/jhb86253817/PIPNet.git", "git clone https://github.com/jhb86253817/PIPNet.git\\ngit -C PIPNet checkout --detach " + PIPNET_REVISION),\n    ]:\n        replace_exact(script, old, new)\n    script.write_text("#!/bin/bash\\nset -euo pipefail\\n" + script.read_text().replace("gdown --id ", "gdown "))\n    replacement = source / "src/pixel3dmm/preprocessing/replacement_code/install_mica_download_flame.sh"\n    replacement.write_text(\'#!/bin/bash\\nset -euo pipefail\\nexec "$DT_PYTHON" "$DT_SESSION/helpers/setup_runtime.py" mica-assets "$DT_SESSION"\\n\')\n    run(["bash", script], root, cwd=source, env=env)\n    assets = source / "src/pixel3dmm/preprocessing/MICA/data"\n    stage_flame(root / "FLAME2020.zip", assets, "2020")\n    if (root / "FLAME2023.zip").exists():\n        stage_flame(root / "FLAME2023.zip", assets, "2023")\n    run([python, "-m", "pip", "check"], root, env=env)\n    run([python, "-c", "import torch, pytorch3d, nvdiffrast.torch, facer, insightface; assert torch.cuda.is_available()"], root, env=env)\n    return python, env\n\n\nif __name__ == "__main__":\n    if sys.argv[1] != "mica-assets":\n        raise ValueError("Unknown installation operation")\n    mica_assets(Path(sys.argv[2]))\n', 'worker.py': '"""User-operated Colab multi-photo fit. Never run on personal data during QA."""\n\nfrom pathlib import Path\nimport json\nimport os\nimport shutil\nimport sys\n\nfrom io_utils import upload_views\nfrom setup_runtime import run\n\n\ndef preprocess(root: Path) -> list:\n    import numpy as np\n    from PIL import Image, ImageOps\n    from insightface.app import FaceAnalysis\n\n    source = root / "pixel3dmm"\n    data = root / "preprocessed"\n    merged = data / "head"\n    views = upload_views(path.name for path in (root / "uploads").iterdir())\n    included, skipped = [], []\n    # Only back may be skipped. Detection/preprocessing failure for a required\n    # view stops the run; it must never silently create a zero-landmark input.\n    detector = FaceAnalysis(name="antelopev2", root=os.environ["DT_INSIGHTFACE_ROOT"],\n                            allowed_modules=["detection"], providers=["CPUExecutionProvider"])\n    detector.prepare(ctx_id=-1, det_size=(512, 512))\n    for view in ("front", "left", "right", "back"):\n        if view not in views:\n            skipped.append({"view": view, "reason": "not_uploaded"})\n            continue\n        input_dir = root / "inputs" / view\n        input_dir.mkdir(parents=True)\n        with Image.open(root / "uploads" / views[view]) as image:\n            normalized = ImageOps.exif_transpose(image).convert("RGB")\n            original_size = list(normalized.size)\n            normalized.save(input_dir / "input.png")\n            # Insightface receives BGR, matching its documented API.\n            if view == "back" and len(detector.get(np.asarray(normalized)[:, :, ::-1].copy())) == 0:\n                skipped.append({"view": view, "reason": "no_face_detected"})\n                continue\n        # run_preprocessing.py uses unchecked os.system. Execute its three\n        # component scripts with checked subprocesses, independently per view.\n        run([sys.executable, source / "scripts/run_cropping.py", "--video_or_images_path", input_dir], root, cwd=source)\n        folder = data / view\n        crop = folder / "cropped/00000.jpg"\n        landmark = folder / "PIPnet_landmarks/00000.npy"\n        bounds = folder / "crop_ymin_ymax_xmin_xmax.npy"\n        for path in (crop, landmark, bounds):\n            if not path.is_file():\n                raise RuntimeError(f"Cropping/landmarks incomplete for {view}")\n        lm = np.load(landmark, allow_pickle=False)\n        if lm.shape != (98, 2) or not np.isfinite(lm).all() or not lm.any():\n            raise RuntimeError(f"Invalid facial landmarks for {view}")\n        run([sys.executable, "demo.py", "-video_name", view, "-a", folder / "arcface"], root,\n            cwd=source / "src/pixel3dmm/preprocessing/MICA")\n        identity = folder / "mica/00000/identity.npy"\n        if not identity.is_file():\n            raise RuntimeError(f"MICA face detection failed for {view}; use a less extreme profile")\n        run([sys.executable, source / "scripts/run_facer_segmentation.py", "--video_name", view], root, cwd=source)\n        segment = folder / "seg_og/00000.png"\n        if not segment.is_file():\n            raise RuntimeError(f"Face segmentation incomplete for {view}")\n        index = len(included)\n        for path, destination in [\n            (crop, merged / f"cropped/{index:05d}.jpg"),\n            (landmark, merged / f"PIPnet_landmarks/{index:05d}.npy"),\n            (segment, merged / f"seg_og/{index:05d}.png"),\n        ]:\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copyfile(path, destination)\n        shutil.copytree(identity.parent, merged / f"mica/{index:05d}")\n        included.append({"view": view, "frame": index, "originalSizeWH": original_size,\n                         "cropBoundsYminYmaxXminXmax": np.load(bounds, allow_pickle=False).tolist()})\n    if len(included) < 3:\n        raise RuntimeError("Front and both profiles must preprocess successfully")\n    (root / "view_map.json").write_text(json.dumps({"included": included, "skipped": skipped}, indent=2))\n    for prediction in ("normals", "uv_map"):\n        run([sys.executable, source / "scripts/network_inference.py",\n             "model.prediction_type=" + prediction, "video_name=head", "viz_uv_mesh=False"], root, cwd=source)\n        for item in included:\n            path = merged / f"p3dmm/{prediction}/{item[\'frame\']:05d}.png"\n            if not path.is_file():\n                raise RuntimeError("Pixel3DMM prediction missing; upstream may have caught an inference error")\n    return included\n\n\ndef fit(root: Path, included: list, config: dict) -> None:\n    import torch\n    from omegaconf import OmegaConf\n    from pixel3dmm.tracking.tracker import Tracker\n\n    source = root / "pixel3dmm"\n    overrides = {\n        "video_name": "head", "num_views": 1, "batch_size": len(included),\n        "iters": config["iters"], "global_iters": config["global_iters"],\n        "is_discontinuous": True, "global_camera": False, "include_neck": False,\n        "use_flame2023": config["flame_version"] == "2023",\n        "ignore_mica": config["flame_version"] == "2023",\n        "w_exp": 0.1, "use_mouth_lmk": False, "w_shape": 0.01, "w_shape_general": 0.001,\n        "normal_super": 2000.0, "sil_super": 1000.0,\n        "save_meshes": True, "save_landmarks": False, "delete_preprocessing": False,\n        "size": 256, "image_size": [256, 256],\n    }\n    cfg = OmegaConf.merge(OmegaConf.load(source / "configs/tracking.yaml"), overrides)\n    tracker = Tracker(cfg)\n    tracker.run()\n    # Explicitly save every view AFTER joint optimization. canonical.ply saved\n    # during online initialization is stale and must not be used as neutral export.\n    for item in included:\n        index = item["frame"]\n        tracker.save_checkpoint(index, selected_frames=torch.tensor([index], device="cuda"))\n    (root / "fit_path.txt").write_text(str(Path(tracker.output_folder)))\n    (root / "fit_config.json").write_text(json.dumps(OmegaConf.to_container(cfg, resolve=True)))\n    tracker.writer.close()\n\n\ndef main(root: Path) -> None:\n    os.environ["DT_WORKER_ACTIVE"] = "1"\n    config = json.loads((root / "config.json").read_text())\n    included = preprocess(root)\n    fit(root, included, config)\n    from export_fit import export_fit\n    export_fit(root, config)\n\n\nif __name__ == "__main__":\n    main(Path(sys.argv[1]))\n', 'export_fit.py': '"""Export fitted identity, neutral geometry, native cameras and alignment overlays."""\n\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport json\nimport shutil\nimport zipfile\n\nimport numpy as np\n\nfrom camera import camera_matrices, projection_matrix\nfrom setup_runtime import PIXEL_REVISION, MICA_REVISION, FACER_REVISION, PIPNET_REVISION\n\n\ndef render_overlay(vertices, faces, camera, image, destination: Path) -> None:\n    import torch\n    import nvdiffrast.torch as dr\n    from PIL import Image\n    import trimesh\n\n    intrinsics, extrinsics, side = camera\n    mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)\n    points = torch.tensor(np.column_stack([vertices, np.ones(len(vertices))]), dtype=torch.float32, device="cuda")\n    matrix = torch.tensor(projection_matrix(intrinsics, side) @ extrinsics, dtype=torch.float32, device="cuda")\n    clips = (points @ matrix.T)[None].contiguous()\n    triangles = torch.tensor(faces.astype(np.int32), device="cuda").contiguous()\n    context = dr.RasterizeCudaContext()\n    raster, _ = dr.rasterize(context, clips, triangles, resolution=[side, side])\n    normals = torch.tensor(np.asarray(mesh.vertex_normals).copy(), dtype=torch.float32, device="cuda")[None]\n    normal_image, _ = dr.interpolate(normals.contiguous(), raster, triangles)\n    shading = (0.35 + 0.65 * normal_image[..., 2:3].abs()).clamp(0, 1)\n    color = shading * torch.tensor([0.2, 0.7, 1.0], device="cuda")\n    mask = (raster[..., 3:4] > 0).float()\n    # The upstream projection matrix has negative fy; array row 0 is already top.\n    rendered = (color * mask)[0].cpu().numpy()\n    alpha = mask[0].cpu().numpy() * 0.5\n    background = np.asarray(Image.open(image).convert("RGB").resize((side, side)), dtype=np.float32) / 255\n    Image.fromarray(np.clip((background * (1 - alpha) + rendered * 0.5) * 255, 0, 255).astype(np.uint8)).save(destination)\n\n\ndef export_fit(root: Path, config: dict) -> None:\n    import torch\n    import trimesh\n    from omegaconf import OmegaConf\n    from pixel3dmm.tracking.flame.FLAME import FLAME\n\n    fit_path = Path((root / "fit_path.txt").read_text())\n    view_map = json.loads((root / "view_map.json").read_text())\n    out = root / "export"\n    (out / "overlays").mkdir(parents=True)\n    (out / "fitted_views").mkdir()\n    checkpoints = [torch.load(fit_path / f"checkpoint/{item[\'frame\']:05d}.frame", map_location="cpu", weights_only=False)\n                   for item in view_map["included"]]\n    shape = np.asarray(checkpoints[0]["flame"]["shape"], dtype=np.float32).reshape(1, -1)\n    if shape.shape != (1, 300) or not np.isfinite(shape).all():\n        raise ValueError("Invalid shared shape coefficients")\n    for checkpoint in checkpoints:\n        if not np.allclose(shape, checkpoint["flame"]["shape"]):\n            raise ValueError("Final per-view checkpoints do not share identity")\n    flame = FLAME(OmegaConf.create({"num_shape_params": 300, "num_exp_params": 100,\n                                   "use_flame2023": config["flame_version"] == "2023"})).cuda()\n    with torch.no_grad():\n        vertices = flame(shape_params=torch.tensor(shape, device="cuda"), cameras=None)[0][0].cpu().numpy()\n    faces = flame.faces.cpu().numpy()\n    neutral = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)\n    neutral.export(out / "head_neutral.obj")\n    neutral.export(out / "head_neutral.ply")\n    identity6d = [1, 0, 0, 0, 1, 0]\n    neutral_parameters = {"shape": shape[0].tolist(), "exp": [0.0] * 100,\n                          "R": identity6d, "t": [0.0] * 3, "neck": identity6d,\n                          "jaw": identity6d, "eyes": identity6d * 2, "eyelids": [0.0, 0.0]}\n    parameters = {"schema": "dt-flame-head-parameters/1", "flameVersion": config["flame_version"],\n                  "rotationEncoding": "6D: first two ROWS of a rotation matrix (Pixel3DMM/PyTorch3D)",\n                  "neutral": neutral_parameters, "views": {}}\n    arrays = {"neutral_" + key: np.asarray(value, dtype=np.float32) for key, value in neutral_parameters.items()}\n    cameras = {"schema": "dt-flame-head-cameras/1", "model": "OpenGL perspective pinhole",\n               "grid": "256x256 independent face crop; pixel centers at (column+0.5,row+0.5)",\n               "projection": "q = worldToCamera @ [x,y,z,1]; depth=-q.z; u=fx*q.x/depth+cx; v=cy-fy*q.y/depth",\n               "meshSpace": "native FLAME metres, unnormalized; extrinsics apply to fitted_views meshes with local neck/jaw/expression already applied",\n               "views": {}, "skipped": view_map["skipped"]}\n    for item, checkpoint in zip(view_map["included"], checkpoints):\n        view, index = item["view"], item["frame"]\n        fitted = {key: np.asarray(value) for key, value in checkpoint["flame"].items()}\n        if not all(np.isfinite(value).all() for value in fitted.values()):\n            raise ValueError("Nonfinite fitted parameters")\n        parameters["views"][view] = {key: value.tolist() for key, value in fitted.items()}\n        arrays.update({view + "_" + key: value for key, value in fitted.items()})\n        joint_transforms = np.asarray(checkpoint["joint_transforms"])\n        arrays[view + "_joint_transforms"] = joint_transforms\n        intrinsics, extrinsics, side = camera_matrices(checkpoint)\n        # A rigid head-centric camera cannot undo nonrigid expression/jaw changes.\n        # Preserve the local neck transform separately, as upstream visualization does.\n        head_centric = extrinsics @ joint_transforms[0, 1]\n        cameras["views"][view] = {\n            **item, "imageSizeWH": [side, side], "intrinsics": intrinsics.tolist(),\n            "worldToCamera": extrinsics.tolist(), "headCentricWorldToCamera": head_centric.tolist(),\n            "headCentricNote": "Use only with the matching fitted mesh after undoing joint_transforms[0,1]; neutral expression changes geometry",\n            "native": {key: np.asarray(value).tolist() for key, value in checkpoint["camera"].items()},\n            "projectionMatrix": projection_matrix(intrinsics, side).tolist(),\n            "overlay": f"overlays/{view}.png", "fittedMesh": f"fitted_views/{view}.ply",\n        }\n        arrays[view + "_intrinsics"] = intrinsics\n        arrays[view + "_world_to_camera"] = extrinsics\n        source_mesh = fit_path / f"mesh/{index:05d}.ply"\n        mesh = trimesh.load(source_mesh, process=False, force="mesh")\n        shutil.copyfile(source_mesh, out / f"fitted_views/{view}.ply")\n        render_overlay(np.asarray(mesh.vertices), np.asarray(mesh.faces), (intrinsics, extrinsics, side),\n                       root / f"preprocessed/head/cropped/{index:05d}.jpg", out / f"overlays/{view}.png")\n    (out / "parameters.json").write_text(json.dumps(parameters, indent=2, allow_nan=False))\n    np.savez_compressed(out / "parameters.npz", **arrays)\n    (out / "cameras.json").write_text(json.dumps(cameras, indent=2, allow_nan=False))\n    provenance = {"createdAt": datetime.now(timezone.utc).isoformat(), "flameVersion": config["flame_version"],\n                  "sources": {"pixel3dmm": PIXEL_REVISION, "MICA": MICA_REVISION, "facer": FACER_REVISION, "PIPNet": PIPNET_REVISION},\n                  "license": "Personal non-commercial research only; Pixel3DMM CC BY-NC 4.0 plus separately accepted FLAME/MICA/insightface/nvdiffrast terms. Not cleared for app distribution.",\n                  "isDiscontinuous": True, "globalCamera": False, "neutralExpression": True, "neutralJaw": True,\n                  "includedViews": [item["view"] for item in view_map["included"]], "skippedViews": view_map["skipped"],\n                  "fitSettings": json.loads((root / "fit_config.json").read_text()),\n                  "limitations": ["No hair reconstruction", "Unseen back anatomy follows FLAME prior", "Scale from model prior; no metric calibration", "Not the rigged twin.glb bundle"]}\n    (out / "provenance.json").write_text(json.dumps(provenance, indent=2, allow_nan=False))\n    with zipfile.ZipFile(root / "head_fit.zip", "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(out.rglob("*")):\n            if path.is_file():\n                archive.write(path, path.relative_to(out).as_posix())\n'}


## Copy FLAME, install, upload, fit and download

Drive authorization is only needed to read the model zip(s). No FLAME
username/password is requested. The credential sections of upstream's
preprocessing installer are replaced in the VM with model-weight downloads.
All subprocess diagnostics stay in the session's `runtime.log`, removed
by cleanup; if installation fails, the cell reports the failing command
without displaying upstream numeric/photo debug output.

Each view is cropped separately before the joint fit. `is_discontinuous=True`
disables temporal smoothness; `global_camera=False` fits intrinsics per photo.
Overlays are at **256x256 on each face crop**, not original-photo coordinates.
Final shared shape coefficients generate the neutral mesh anew after joint
fitting: expression/eyelids zero, jaw/neck/eyes/global rotation identity,
global translation zero. Do not use upstream's earlier canonical mesh.

Enable browser downloads. The cell waits for `files.download()` to transfer
the archive into browser memory before deleting the VM source. It cannot
verify that you saved the browser download to disk.


In [ ]:
from pathlib import Path
import gc
import importlib
import json
import shutil
import sys
import tempfile
from google.colab import drive, files

def run_session():
    session = Path(tempfile.mkdtemp(prefix="dt-pixel3dmm-session-", dir="/content"))
    helpers = session / "helpers"
    helpers.mkdir()
    uploaded = {}
    mounted = False
    config_created = False
    env_file = Path.home() / ".config/pixel3dmm/.env"
    try:
        for name, source in EMBEDDED_FILES.items():
            (helpers / name).write_text(source, encoding="utf-8")
        sys.path.insert(0, str(helpers))
        importlib.invalidate_caches()
        from io_utils import upload_views, download_and_wait
        from setup_runtime import install, run
        selected = {FLAME_VERSION: FLAME_ZIP_DRIVE_PATH}
        if FLAME_VERSION == "2023":
            selected["2020"] = FLAME2020_ZIP_DRIVE_PATH
        drive.mount("/content/drive")
        mounted = True
        for version, filename in selected.items():
            model_zip = Path(filename).resolve()
            if not model_zip.is_relative_to(Path("/content/drive").resolve()) or model_zip.suffix.lower() != ".zip":
                raise ValueError("Select a FLAME zip inside /content/drive")
            if not model_zip.is_file():
                raise FileNotFoundError("FLAME zip not found; check the parameter cell")
            # Read-only Drive usage: this is the ONLY Drive file operation.
            shutil.copyfile(model_zip, session / ("FLAME" + version + ".zip"))
        drive.flush_and_unmount()
        mounted = False
        if env_file.exists():
            raise RuntimeError("Existing Pixel3DMM config found. Use a fresh Colab runtime")
        env_file.parent.mkdir(parents=True, exist_ok=True)
        paths = {
            "PIXEL3DMM_CODE_BASE": str(session / "pixel3dmm"),
            "PIXEL3DMM_PREPROCESSED_DATA": str(session / "preprocessed"),
            "PIXEL3DMM_TRACKING_OUTPUT": str(session / "tracking"),
        }
        env_file.write_text("".join(key + "=" + json.dumps(value) + "\n" for key, value in paths.items()), encoding="utf-8")
        config_created = True
        python, worker_env = install(session, GPU_ARCH)
        worker_env.update(paths)
        (session / "config.json").write_text(json.dumps({
            "flame_version": FLAME_VERSION, "iters": ITERS, "global_iters": GLOBAL_ITERS,
        }), encoding="utf-8")
        uploads = session / "uploads"
        uploads.mkdir()
        print("Upload de-glassed front, left, right and back head photos together.")
        uploaded = files.upload(target_dir=str(uploads))
        upload_views(uploaded.keys())
        uploaded.clear()
        print("Preprocessing each view, then fitting shared identity with independent cameras.")
        run([python, helpers / "worker.py", session], session, cwd=session / "pixel3dmm", env=worker_env)
        download_and_wait(session / "head_fit.zip")
        print("head_fit.zip transferred to your browser. Save it to user-data/twin/head/flame/.")
    finally:
        uploaded.clear()
        # Nested finally ensures private files are removed even if Drive unmount fails.
        try:
            if mounted:
                drive.flush_and_unmount()
        finally:
            try:
                if config_created:
                    env_file.unlink(missing_ok=True)
            finally:
                if session.resolve().parent != Path("/content") or not session.name.startswith("dt-pixel3dmm-session-"):
                    raise ValueError("Refusing unsafe session cleanup")
                shutil.rmtree(session)
                if str(helpers) in sys.path:
                    sys.path.remove(str(helpers))
                for name in EMBEDDED_FILES:
                    sys.modules.pop(Path(name).stem, None)
                gc.collect()
                print("Privacy cleanup: VM uploads, FLAME, weights, outputs, logs and archive removed.")

run_session()


## Privacy cleanup (last cell)

The session `finally` already deletes all VM photos, derived files, model
archives/extracted FLAME and weights on success or ordinary errors.
This final cell checks for leftovers, including after a previous interrupted
run. If Run all stopped on an error, run this cell manually.
No Drive file is deleted. Always disconnect and delete the runtime after saving.


In [ ]:
from pathlib import Path
import gc
import shutil

for leftover in Path("/content").glob("dt-pixel3dmm-session-*"):
    if leftover.resolve().parent != Path("/content") or leftover.is_symlink():
        raise ValueError("Refusing unsafe cleanup path")
    shutil.rmtree(leftover)
# Remove only this notebook's session-bound configuration, never an unrelated config.
config_file = Path.home() / ".config/pixel3dmm/.env"
if config_file.exists() and '"/content/dt-pixel3dmm-session-' in config_file.read_text():
    config_file.unlink()
gc.collect()
print("No Pixel3DMM session files remain. Save your download, then disconnect and delete runtime.")
